In [ ]:
# 必要なら最初に実行
# Colabで geopandas / pyogrio が未インストールの場合だけコメントアウトを外してください。
# !pip -q install geopandas pyogrio

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ============================================================
# Figure 4 revised input settings
# 14_v5の同一Extra Trees strict first-occurrence評価結果を使う。
# ファイル探索はしない。必要ファイルをこのノートブックと同じ場所に置くか、下のPathを直接変更する。
# ============================================================

CASE_RESULTS_PATH = Path("14_v5_case_results_same_extratrees_leakage_safe.csv")

# case_resultsには通常grid_lat/grid_lonが入っていないため、座標取得用のpanelまたはgrid座標表を指定する。
# 例1: panel parquetを使う場合
COORDS_PATH = Path("hpai_weekly_grid_panel_with_spatiotemporal_features_v4_strict_06_compatible.parquet")

# 例2: grid_id, grid_lat, grid_lonだけを含むCSVを使う場合は、上の行をコメントアウトして下を使う。
# COORDS_PATH = Path("grid_coordinates.csv")

OUTPUT_DIR = Path(".")
BASELINE_MODEL = "baseline_same_extratrees_geo_season_weather"

EXPECTED_TOTAL = 49
EXPECTED_TOP10_SUCCESS = 11
EXPECTED_TOP10_MISS = 38


In [ ]:
# ============================================================
# Figure 4: Spatial distribution of Top 10% successes and misses
# Final v7 baseline Extra Trees strict evaluation
# ============================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

candidate_paths = [
    Path(
        "/content/drive/MyDrive/avian_influenza_project/processed/"
        "model_outputs_riskmap_eval/"
        "14_v7_baseline_top10_success_miss_cases.csv"
    ),
    Path("14_v7_baseline_top10_success_miss_cases.csv"),
]

CASE_RESULTS_PATH = next(
    (path for path in candidate_paths if path.exists()),
    None,
)

if CASE_RESULTS_PATH is None:
    raise FileNotFoundError(
        "14_v7_baseline_top10_success_miss_cases.csv was not found.\n"
        "Checked:\n"
        + "\n".join(str(path) for path in candidate_paths)
    )

OUTPUT_DIR = CASE_RESULTS_PATH.parent

df = pd.read_csv(CASE_RESULTS_PATH)

required_cols = {
    "grid_id",
    "grid_lat",
    "grid_lon",
    "top10_group",
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}\n"
        f"Available columns: {df.columns.tolist()}"
    )

plot_df = df.dropna(
    subset=[
        "grid_id",
        "grid_lat",
        "grid_lon",
        "top10_group",
    ]
).copy()

expected_counts = {
    "Top 10 success": 11,
    "Top 10 miss": 38,
}

observed_counts = (
    plot_df["top10_group"]
    .value_counts()
    .reindex(expected_counts.keys(), fill_value=0)
    .to_dict()
)

print("Using:", CASE_RESULTS_PATH)
print("Observed counts:", observed_counts)

if observed_counts != expected_counts:
    raise ValueError(
        "The category counts do not match the final v7 results.\n"
        f"Expected: {expected_counts}\n"
        f"Observed: {observed_counts}"
    )

if len(plot_df) != 49:
    raise ValueError(
        f"Expected 49 events, but found {len(plot_df)}."
    )

map_ready_path = (
    OUTPUT_DIR
    / "figure4_map_ready_top10_success_miss_14v7.csv"
)
plot_df.to_csv(
    map_ready_path,
    index=False,
    encoding="utf-8-sig",
)
print("Saved map-ready data:", map_ready_path)

gdf_points = gpd.GeoDataFrame(
    plot_df,
    geometry=gpd.points_from_xy(
        plot_df["grid_lon"],
        plot_df["grid_lat"],
    ),
    crs="EPSG:4326",
)

countries_url = (
    "https://naturalearth.s3.amazonaws.com/"
    "10m_cultural/ne_10m_admin_0_countries.zip"
)
world = gpd.read_file(countries_url)
japan = world[
    world["ADMIN"] == "Japan"
].copy()

admin1_url = (
    "https://naturalearth.s3.amazonaws.com/"
    "10m_cultural/ne_10m_admin_1_states_provinces.zip"
)
admin1 = gpd.read_file(admin1_url)

if "adm0_name" in admin1.columns:
    japan_admin1 = admin1[
        admin1["adm0_name"] == "Japan"
    ].copy()
elif "admin" in admin1.columns:
    japan_admin1 = admin1[
        admin1["admin"] == "Japan"
    ].copy()
else:
    japan_admin1 = gpd.GeoDataFrame(
        geometry=[],
        crs="EPSG:4326",
    )

category_order = [
    "Top 10 success",
    "Top 10 miss",
]

label_map = {
    "Top 10 success": "Top 10% success",
    "Top 10 miss": "Top 10% miss",
}

color_map = {
    "Top 10 success": "#2ca02c",
    "Top 10 miss": "#d62728",
}

marker_map = {
    "Top 10 success": "o",
    "Top 10 miss": "s",
}

fig, ax = plt.subplots(
    figsize=(7.5, 8.8)
)

japan.plot(
    ax=ax,
    color="white",
    edgecolor="black",
    linewidth=0.8,
    zorder=1,
)

if len(japan_admin1) > 0:
    japan_admin1.boundary.plot(
        ax=ax,
        color="lightgray",
        linewidth=0.4,
        zorder=2,
    )

for category in category_order:
    sub = gdf_points[
        gdf_points["top10_group"] == category
    ]
    sub.plot(
        ax=ax,
        marker=marker_map[category],
        color=color_map[category],
        edgecolor="black",
        linewidth=0.4,
        markersize=65,
        label=(
            f"{label_map[category]} "
            f"(n = {len(sub)})"
        ),
        zorder=5,
    )

ax.set_xlim(128.5, 146.5)
ax.set_ylim(30.0, 46.0)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax.legend(
    title="Category",
    loc="upper left",
    frameon=True,
    fontsize=9,
    title_fontsize=9,
)

ax.set_aspect(
    "equal",
    adjustable="box",
)
ax.grid(
    True,
    linewidth=0.3,
    alpha=0.4,
)

plt.tight_layout()

png_path = (
    OUTPUT_DIR
    / "figure4_top10_success_miss_map_14v7.png"
)
pdf_path = (
    OUTPUT_DIR
    / "figure4_top10_success_miss_map_14v7.pdf"
)

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    pdf_path,
    bbox_inches="tight",
)
plt.show()

print("Saved:", png_path)
print("Saved:", pdf_path)
